<a href="https://colab.research.google.com/github/07ashu97912-hub/AI-Based-Smart-Cab-Allocation-and-Driver-Repositioning-System/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# importing datasets and learning about data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/ml data/uber-raw-data-apr14.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\n summary:")
print(df.describe())

# converting the required column to datetime

In [ ]:
df["Date/Time"] = pd.to_datetime(df["Date/Time"])
print(df.dtypes)
print(df["Date/Time"].head())

# understanding coordinates to prevent unnecessary clusters and noise

In [ ]:
print("Smallest Latitudes:")
print(df.nsmallest(10, "Lat")[["Date/Time", "Lat", "Lon"]])

print("\nLargest Latitudes:")
print(df.nlargest(10, "Lat")[["Date/Time", "Lat", "Lon"]])

print("\nSmallest Longitudes:")
print(df.nsmallest(10, "Lon")[["Date/Time", "Lat", "Lon"]])

print("\nLargest Longitudes:")
print(df.nlargest(10, "Lon")[["Date/Time", "Lat", "Lon"]])

# choosing new york as our study area

In [ ]:
nyc_area = (
    (df["Lat"] >= 40.4) &
    (df["Lat"] <= 41.0) &
    (df["Lon"] >= -74.25) &
    (df["Lon"] <= -73.7)
)

print("Inside study area:", nyc_area.sum())
print("Outside study area:", (~nyc_area).sum())

print("outside area percentage:",
      round((~nyc_area).mean() * 100, 2))

In [26]:
df_ny = df[nyc_area].copy()

df_ny.reset_index(drop=True, inplace=True)

print("Original dataset:", df.shape)
print("Clean dataset:", df_ny.shape)

print("\nRemaining coordinate range:")
print("Latitude:", df_ny["Lat"].min(), "to", df_ny["Lat"].max())
print("Longitude:", df_ny["Lon"].min(), "to", df_ny["Lon"].max())

Original dataset: (564516, 4)
Clean dataset: (562496, 4)

Remaining coordinate range:
Latitude: 40.4277 to 40.9998
Longitude: -74.25 to -73.7


# handling duplicates records

In [ ]:
print("Total duplicate rows:", df_ny.duplicated().sum())

duplicates = df_ny[df_ny.duplicated(keep=False)]

print("\nSample duplicate records:")
print(duplicates.sort_values(
    ["Date/Time", "Lat", "Lon"]
).head(20))

In [ ]:
duplicate_groups = (
    df_ny
    .groupby(["Date/Time", "Lat", "Lon"])
    .size()
    .sort_values(ascending=False)
)

print("\nMost repeated timestamp-location combinations:")
print(duplicate_groups.head(20))

In [ ]:
df_ny = df_ny.drop_duplicates().reset_index(drop=True)

print("Dataset after removing duplicates:", df_ny.shape)
print("Remaining duplicates:", df_ny.duplicated().sum())

# visualization the pickup locations

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

plt.scatter(
    df_ny["Lon"],
    df_ny["Lat"],
    s=1,
    alpha=0.3
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Uber Pickup Locations - NYC, April 2014")

plt.show()